### Full Voice Pipeline — AgriVoice

Complete voice assistant for Nigerian farmers.

## Architecture

🎤 microphone
   ↓ VAD (silero-vad)
📝 STT (faster-whisper, local)
   ↓
🌍 translate → English (GPT + terminology)
   ↓
📚 RAG (Chroma)
   ↓
💬 answer (GPT)
   ↓
🌍 translate → user's language
   ↓
🔊 TTS (YarnGPT with OpenAI fallback)
   ↓
🔊 speaker

## Supported languages
English, Yoruba, Hausa, Igbo, Nigerian Pidgin

In [1]:

import os
import time
import uuid
import requests

from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from faster_whisper import WhisperModel
from silero_vad import load_silero_vad

import torch
import numpy as np
import sounddevice as sd
import soundfile as sf

c:\Users\USER\rag_course\rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load .env
load_dotenv()

# OpenAI client — used by translation and fallback TTS
client = OpenAI()

# YarnGPT key — used by primary TTS
YARNGPT_KEY = os.getenv('YARNGPT_API_KEY')

print(f'   OpenAI key:  {"loaded" if client else "missing"}')
print(f'   YarnGPT key: {"loaded" if YARNGPT_KEY else "missing"}')


   OpenAI key:  loaded
   YarnGPT key: loaded


load faster-whisper (local STT)

In [3]:
# Load the local STT model
stt_model = WhisperModel('base', device='cpu', compute_type='int8')

load RAG chain + retriever

In [4]:
# LLM used by the RAG chain
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Embeddings for the Chroma vector store
embeddings = OpenAIEmbeddings()

# Load the domain knowledge base
vectorstore = Chroma(
    persist_directory=r'C:\Users\USER\rag_course\chroma_db_domain',
    embedding_function=embeddings,
)
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})

# RAG prompt — short answers because they'll be spoken aloud
rag_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are AgriVoice, a farming assistant for Nigerian farmers. '
     'Answer using ONLY the context below. Keep it under 3 sentences. '
     'If the answer is not in the context, say "I don\'t know".'),
    ('human', 'Context:\n{context}\n\nQuestion: {question}'),
])

rag_chain = rag_prompt | llm | StrOutputParser()

print(f'✅ RAG ready — {vectorstore._collection.count()} chunks')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ RAG ready — 82 chunks


load terminology store

In [5]:
# Terminology Chroma store — correct crop/disease translations
terminology_store = Chroma(
    persist_directory=r'C:\Users\USER\rag_course\chroma_db_terminology',
    embedding_function=embeddings,
    collection_name='terminology',
)
terminology_retriever = terminology_store.as_retriever(search_kwargs={'k': 5})

print(f'✅ Terminology ready — {terminology_store._collection.count()} docs')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Terminology ready — 36 docs


translation helpers

In [6]:
# Retrieve relevant crop/disease terminology from Chroma
def get_relevant_terms(text: str) -> str:
    docs = terminology_retriever.invoke(text)
    return '\n'.join(f'- {d.page_content}' for d in docs)


# Translate English text to a Nigerian language using GPT + terminology hints
def translate(text: str, target_lang: str) -> str:
    if target_lang == 'English':
        return text

    terms = get_relevant_terms(text)

    system = (
        f'You are an expert translator for Nigerian languages.\n'
        f'Translate the following English text to {target_lang}.\n\n'
        f'MANDATORY VOCABULARY (use these exact words):\n{terms}\n\n'
        f'RULES:\n'
        f'1. Output ONLY {target_lang} — no English mixed in, except proper nouns.\n'
        f'2. Use the correct tone marks and diacritics.\n'
        f'3. Output ONLY the translation. No explanations.\n'
    )

    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': text},
        ],
        temperature=0,
    )
    return response.choices[0].message.content.strip()


# Translate FROM a Nigerian language TO English using GPT + terminology hints
def translate_to_english(text: str, source_lang: str) -> str:
    if source_lang == 'English':
        return text

    terms = get_relevant_terms(text)

    system = (
        f'You are an expert translator for Nigerian languages.\n'
        f'Translate the following {source_lang} text to English.\n\n'
        f'MANDATORY VOCABULARY (use these to decode crops/diseases):\n{terms}\n\n'
        f'RULES:\n'
        f'1. Output ONLY English.\n'
        f'2. Use the terminology above to correctly identify crops.\n'
        f'3. Output ONLY the translation. No explanations.\n'
    )

    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': text},
        ],
        temperature=0,
    )
    return response.choices[0].message.content.strip()


print('Translation helpers ready')

Translation helpers ready


load silero-vad (voice activity detection)

In [7]:
# VAD model — detects when the farmer starts and stops speaking
vad_model = load_silero_vad()

print('silero-vad ready')

c:\Users\USER\rag_course\rag_env\Lib\site-packages\torch\jit\_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


silero-vad ready


TTS module (YarnGPT + OpenAI fallback)

In [8]:
# Map our language names -> YarnGPT voice names
YARNGPT_VOICES = {
    'English': 'idera',
    'Yoruba': 'idera',
    'Igbo': 'chinenye',
    'Hausa': 'umar',
    'Nigerian Pidgin': 'tayo',
}

# Generate speech with YarnGPT
def yarngpt_tts(text: str, voice: str) -> bytes:
    submit = requests.post(
        'https://yarngpt.ai/api/v1/tts',
        headers={
            'Authorization': f'Bearer {YARNGPT_KEY}',
            'Content-Type': 'application/json',
            'Idempotency-Key': str(uuid.uuid4()),
        },
        json = {'text': text, 'voice': voice},
        timeout=30,
    )
    
    if submit.status_code != 202:
        raise RuntimeError(f'YarnGPT submit failed: {submit.status_code}')
    
    job_id = submit.json()['job_id']
    
    for _ in range(30):
        time.sleep(2)
        status = requests.get(
            f'https://yarngpt.ai/api/v1/status/{job_id}',
            headers={'Authorization': f'Bearer {YARNGPT_KEY}'},
            timeout=30
        )
        data = status.json()
        state = data.get('status')
        
        if state == 'completed':
            audio = requests.get(data['audio_url'], timeout=30)
            return audio.content
        
        if state == 'failed':
            raise RuntimeError(f'YarnGPT job failed: {data}')
        
    raise TimeoutError('YarnGPT job timed out')


# Production TTS: YarnGPT first, OpenAI fallback
def tts_speak(text: str, language: str = 'English') -> bytes:
    # ─── PART 1: YarnGPT (primary) ───
    try:
        voice = YARNGPT_VOICES.get(language, 'idera')
        print(f'   🔊 YarnGPT ({voice})...')
        return yarngpt_tts(text, voice)
    
    except Exception as e:
        print(f'     YarnGPT failed: {e}')
        print('   🔊 Falling back to OpenAI TTS...')
        
    # ─── PART 2: OpenAI (fallback) ───
    response = client.audio.speech.create(
        model='tts-1',
        voice='alloy',
        input=text
    )
    return response.content

print('TTS module ready (YarnGPT + OpenAI fallback)')
    

TTS module ready (YarnGPT + OpenAI fallback)


VoicePipeline

In [13]:
class VoicePipeline:
    '''
    One voice turn: audio in -> STT -> RAG -> TTS -> audio out.
    '''
    
    # Store config
    def __init__(self, language: str = 'English'):
        self.language = language
        print(f'VoicePipeline created (language={self.language})')
    
    # Record from mic, stop automatically when VAD detects silence
    def record_until_silence(
        self,
        max_seconds: int= 15,
        silence_seconds: float = 1.5,
        min_speech_seconds: float = 0.5,
        device: int = 1,
    ) -> str:
        
        # Whisper and VAD both expect 16 kHz
        sample_rate = 16000
        
        # 512 samples per chunk = 32 ms of audio per VAD check
        chunk_samples = 512
        
        print(f'🎤 Recording (stop after {silence_seconds}s of silence)...')
        
        # Store and capture every chunk
        chunks = []
        silence_duration = 0.0
        speech_duration = 0.0
        start_time = time.time()
        
        # Open the mic stream
        with sd.InputStream(
            samplerate=sample_rate,
            channels=1,   # mono
            dtype='int16',
            device=device,    # mic name
            blocksize=chunk_samples     # 32 ms per read
        ) as stream:
            while True:
                # Read one 32 ms chunk from the mic
                chunk, _ = stream.read(chunk_samples)
                # Save it
                chunks.append(chunk)
                
                # Convert int16 chunk → float32 in range -1.0 to 1.0 (what VAD expects)
                audio_chunk = chunk.flatten().astype(np.float32) / 32768.0
                
                # Wrap in a torch tensor (silero-vad uses PyTorch)
                tensor = torch.from_numpy(audio_chunk)
                
                # Ask VAD: probability this chunk has speech (0.0 to 1.0)
                prob = vad_model(tensor, sample_rate).item()
                
                # Threshold: above 0.5 = speech, below = silence
                if prob > 0.5:
                    # Add this chunk's duration to speech time
                    speech_duration += chunk_samples / sample_rate
                    # Reset silence counter 
                    silence_duration = 0.0
                else:
                    # Add this chunk's duration to silence time
                    silence_duration += chunk_samples / sample_rate
                
                # Recording total time    
                elapsed = time.time() - start_time
                
                # Stop condition 1: heard enough speech AND been silent long enough
                if speech_duration >= min_speech_seconds and silence_duration >= silence_seconds:
                    print(f'End of speech ({elapsed:.2f}s)')
                    break
                
                if elapsed >= max_seconds:
                    print(f'Max duration: ({max_seconds}s)')
                    break
            
        # Stack all chunks into one continuous audio array    
        audio = np.concatenate(chunks, axis=0)
        
        # save path
        out_path = r'C:\Users\USER\rag_course\12b_voice_agents\data\mic_live.wav'
        
        # Write WAV file to disk
        sf.write(out_path, audio, sample_rate)
        print(f' {out_path}')
        return out_path
    
    # Transcribe an audio file to text using local faster-whisper
    def listen(self, audio_path: str) -> str:
        
        # faster-whisper returns a generator
        
        segments, info = stt_model.transcribe(audio_path, beam_size=5)
        
        # iterate to get all segments
        # Join all segment texts into one string
        return ' '.join(seg.text.strip() for seg in segments)
    
    # Translate to English → retrieve → answer → translate back to user's language
    def think(self, question: str) -> str:
        
        # Step 1: translate the farmer's question to English
        question_en = translate_to_english(question, self.language)
        
        # Step 2: retrieve relevant docs from Chroma
        docs = retriever.invoke(question_en)
        
        # Concatenate the page text of each retrieved doc
        context = '\n\n'.join(doc.page_content for doc in docs)
        
        # Step 3: run the RAG chain — takes context + question, returns English answer
        answer_en = rag_chain.invoke({
            'context': context,
            'question': question_en
        })
        
        # Step 4: translate the English answer back to the farmer's language
        return translate(answer_en, self.language)
    
    # Speak the answer using YarnGPT with OpenAI fallback
    def speak(self, answer: str) -> bytes:
        return tts_speak(answer, language=self.language)
    
    # Chain listen → think → speak — the full one-turn voice flow
    def turn(self, audio_path: str) -> bytes:
        
        # Stage 1: STT
        print('🎧 Listening...')
        question = self.listen(audio_path)
        print(f'   📝 {question}')

        # Stage 2: RAG + translation
        print('🧠 Thinking...')
        answer = self.think(question)
        print(f'   💬 {answer}')
        
        # Stage 3: TTS
        print('🔊 Speaking...')
        # Hand the audio bytes back to the caller
        return self.speak(answer)
    
    # Full mic-to-speaker turn using VAD-driven recording
    def conversation_turn(self) -> None:
        # Local import — only needed for display inside notebooks
        from IPython.display import Audio, display

        # Record from mic until silence detected
        audio_path = self.record_until_silence()
        # Run the full pipeline
        audio_bytes = self.turn(audio_path)

        # Save the answer audio to disk
        out_path = r'C:\Users\USER\rag_course\12b_voice_agents\data\answer_live.mp3'
        with open(out_path, 'wb') as f:
            f.write(audio_bytes)
        print(f'💾 {out_path}')

        # Play it in the notebook
        display(Audio(out_path))


# Confirm the class is defined
print('✅ VoicePipeline class defined')

✅ VoicePipeline class defined


Test the pipeline in English

In [16]:
# Create a pipeline instance configured for English
vp = VoicePipeline(language='English')

# Run one full mic-to-speaker turn:
#   🎤 record (VAD stops when you finish speaking)
#   📝 STT (faster-whisper)
#   🧠 translate → English (no-op) → RAG → answer → translate back (no-op)
#   🔊 YarnGPT speaks it (or OpenAI TTS if YarnGPT fails)
vp.conversation_turn()

VoicePipeline created (language=English)
🎤 Recording (stop after 1.5s of silence)...
End of speech (5.28s)
 C:\Users\USER\rag_course\12b_voice_agents\data\mic_live.wav
🎧 Listening...
   📝 How can it treat cassava disease?
🧠 Thinking...
   💬 To treat cassava disease, use disease-free cuttings for planting, plant resistant varieties, remove and destroy infected plants immediately, and control whitefly vectors with appropriate insecticides.
🔊 Speaking...
   🔊 YarnGPT (idera)...
💾 C:\Users\USER\rag_course\12b_voice_agents\data\answer_live.mp3


Test in Yoruba 

In [17]:
# Create a pipeline configured for Yoruba
vp = VoicePipeline(language='Yoruba')


# Speak in English into the mic
# The pipeline will translate the answer to Yoruba and speak it back
vp.conversation_turn()

VoicePipeline created (language=Yoruba)
🎤 Recording (stop after 1.5s of silence)...
End of speech (5.45s)
 C:\Users\USER\rag_course\12b_voice_agents\data\mic_live.wav
🎧 Listening...
   📝 How can I treat cassava disease?
🧠 Thinking...


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


   💬 Lati tọju àrùn mosaic ẹgẹ́, lo awọn gige ti ko ni àrùn fun ikore, gbin awọn irugbin ẹgẹ́ ti o ni resistance, ki o si yọkuro ki o si pa awọn ọgbin ti o ni àrùn lẹsẹkẹsẹ. Ṣakoso kòkòrò funfun pẹlu awọn insecticides to yẹ ki o si yago fun gbin ẹgẹ́ nitosi awọn ilẹ ti o ni àrùn.
🔊 Speaking...
   🔊 YarnGPT (idera)...
💾 C:\Users\USER\rag_course\12b_voice_agents\data\answer_live.mp3


In [19]:
# Create a pipeline configured for Hausa
vp = VoicePipeline(language='Hausa')


# Speak in English into the mic
# The pipeline will translate the answer to Hausa and speak it back
vp.conversation_turn()

VoicePipeline created (language=Hausa)
🎤 Recording (stop after 1.5s of silence)...
End of speech (5.13s)
 C:\Users\USER\rag_course\12b_voice_agents\data\mic_live.wav
🎧 Listening...
   📝 How can I treat cassava disease?
🧠 Thinking...
   💬 Don magance cutar mosaic na rogo, yi amfani da itatuwa masu lafiya don shuka, shuka nau'ikan rogo masu juriya, kuma a cire da lalata shukar da ta kamu da cutar nan take. Sarrafa farar kwari tare da magungunan kashe kwari masu dacewa kuma a guji shuka rogo kusa da filayen da suka kamu da cutar.
🔊 Speaking...
   🔊 YarnGPT (umar)...
💾 C:\Users\USER\rag_course\12b_voice_agents\data\answer_live.mp3


In [21]:
# Create a pipeline configured for Igbo
vp = VoicePipeline(language='Igbo')

# Speak in English into the mic
# The pipeline will translate the answer to Igbo and speak it back
vp.conversation_turn()

VoicePipeline created (language=Igbo)
🎤 Recording (stop after 1.5s of silence)...
End of speech (4.49s)
 C:\Users\USER\rag_course\12b_voice_agents\data\mic_live.wav
🎧 Listening...
   📝 How can I treat cassava disease?
🧠 Thinking...
   💬 Iji wụnye ọrịa mosaic akpụ, jiri mkpụrụ osisi na-enweghị ọrịa maka ịkụ, kụọ ụdị akpụ nke na-eguzogide, ma wepụ ma bibie osisi ndị metụtara ya ozugbo. Jikwaa ijiji ọcha na ọgwụ mgbochi insects kwesịrị ekwesị ma zere ịkụ akpụ nso na ubi ndị metụtara.
🔊 Speaking...
   🔊 YarnGPT (chinenye)...
💾 C:\Users\USER\rag_course\12b_voice_agents\data\answer_live.mp3
